In [2]:
from ler import LeR
import numpy as np

In [3]:
ler = LeR(
    spin_precession=True,
)


Initializing LeR class...


Initializing LensGalaxyParameterDistribution class...


Initializing OpticalDepth class

comoving_distance interpolator will be loaded from ./interpolator_json/comoving_distance/comoving_distance_0.json
angular_diameter_distance interpolator will be loaded from ./interpolator_json/angular_diameter_distance/angular_diameter_distance_0.json
angular_diameter_distance interpolator will be loaded from ./interpolator_json/angular_diameter_distance/angular_diameter_distance_0.json
differential_comoving_volume interpolator will be loaded from ./interpolator_json/differential_comoving_volume/differential_comoving_volume_0.json
using ler available velocity dispersion function : velocity_dispersion_ewoud
velocity_dispersion_ewoud interpolator will be loaded from ./interpolator_json/velocity_dispersion/velocity_dispersion_ewoud_0.json
using ler available axis_ratio function : rayleigh
rayleigh interpolator will be loaded from ./interpolator_json/axis_ratio/rayleigh_2.j

In [4]:
# function to convert Dl to zs
# ler.luminosity_distance.function_inverse(np.array([1000.,1001]))

In [8]:
size = 1 # make sure to set this to the number of samples you want to generate
# replace this with your posterior samples
# unlensed_params should be a dictionary with keys 'zs', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance'
zs = np.array([1.0])
unlensed_params = ler.sample_gw_parameters(size)
unlensed_params['zs'] = zs

# use only these keys in unlensed_params 
# 'zs', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance'
unlensed_params = {k: unlensed_params[k] for k in ['zs', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance']}
unlensed_params.keys()

dict_keys(['zs', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance'])

In [9]:
# mass_1_source = mass_1/(1+zs)

In [10]:
# how to create dictionary of unlensed parameters for testing
# unlensed_params = dict(
#     zs=np.array([1.0, 2.0]),
#     mass_1_source = np.array([30.0, 35.0]),
#     ....
# )

In [11]:
# lens parameters 
zl = ler.lens_redshift_sl.rvs(size, unlensed_params['zs'])
sigma, q, phi, gamma, gamma1, gamma2 = ler.cross_section_based_sampler(
    size, unlensed_params['zs'], zl
)

lens_parameters = {
    "zl": zl,
    "zs": unlensed_params['zs'],
    "sigma": sigma,
    "theta_E": ler.compute_einstein_radii(sigma, zl, unlensed_params['zs']),
    "q": q,
    "phi": phi,
    "gamma": gamma,
    "gamma1": gamma1,
    "gamma2": gamma2,
}
lens_parameters.update(unlensed_params) # combine unlensed and lens parameters into one dictionary
lens_parameters.keys()

dict_keys(['zl', 'zs', 'sigma', 'theta_E', 'q', 'phi', 'gamma', 'gamma1', 'gamma2', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance'])

In [12]:
# image parameteres
image_params = ler.image_properties_function(lens_parameters)

computing image properties using ler's epl+shear (analytical, njit) solver...


In [13]:
# image_params = ler.recover_redundant_parameters(image_params)

In [14]:
image_params = ler.produce_effective_params(image_params)

In [15]:
image_params.keys()

dict_keys(['zl', 'zs', 'sigma', 'q', 'phi', 'gamma', 'gamma1', 'gamma2', 'geocent_time', 'ra', 'dec', 'phase', 'luminosity_distance', 'x0_image_positions', 'x1_image_positions', 'magnifications', 'time_delays', 'image_type', 'n_images', 'x_source', 'y_source', 'effective_luminosity_distance', 'effective_geocent_time', 'effective_phase', 'effective_ra', 'effective_dec'])

In [16]:
image_params['luminosity_distance'][0], image_params['effective_luminosity_distance'][0]

(np.float64(21441.8198005838),
 array([12481.37120387, 21507.52400295,            nan,
                   nan]))

In [17]:
image_params['luminosity_distance']

array([21441.81980058])

In [18]:
#1st image
image_params['effective_luminosity_distance'][:,0]

array([12481.37120387])

In [19]:
#2nd image
image_params['effective_luminosity_distance'][:,1]

array([21507.52400295])

In [24]:
image_params['time_delays'][0]+ unlensed_params['geocent_time']

array([1.24726028e+09, 1.24823974e+09,            nan,
                  nan])

In [25]:
unlensed_params['luminosity_distance']*np.sqrt(np.abs(image_params['magnifications'][0]))

array([36835.02628447, 21376.31632064,            nan,
                  nan])

In [27]:
# image_type[image_type == 1.0] = 0.0
# image_type[image_type == 2.0] = np.pi / 2.0
# image_type[image_type == 3.0] = np.pi
image_params['image_type'][0]

array([1, 2, 0, 0])